In [1]:
import pandas as pd

In [2]:
results_df = pd.read_csv("../output/results_all_folds.tsv", sep="\t")
results_df.head()

,eval_loss,eval_model_preparation_time,eval_bleu,eval_chrf,eval_ter,eval_runtime,eval_samples_per_second,eval_steps_per_second,model,fold
0,1.477993,0.0023,45.696161,67.150245,45.468782,133.3450,28.070,1.755,Helsinki-NLP/opus-mt-es-en,fold_1
1,1.718562,0.0045,39.064323,61.928381,51.640536,492.4575,7.601,0.475,facebook/m2m100_418M,fold_1
2,1.560524,0.0088,40.725904,63.178648,48.893257,888.4807,4.213,0.263,facebook/m2m100_1.2B,fold_1
3,1.252902,0.0047,41.606861,64.923934,45.783595,444.3771,8.423,0.527,facebook/nllb-200-distilled-600M,fold_1
4,1.893300,0.0023,44.023599,64.947178,48.402677,138.7123,28.866,1.810,Helsinki-NLP/opus-mt-es-en,fold_2


In [3]:
metric_cols = ["eval_bleu","eval_chrf","eval_ter","eval_runtime"]

agg = results_df.groupby("model")[metric_cols].agg(["mean","std"])

# summary con el MISMO índice (model)
summary = pd.DataFrame(index=agg.index)

for metric in metric_cols:
    summary[metric] = (
        agg[(metric, "mean")].round(2).astype(str)
        + " ± "
        + agg[(metric, "std")].round(2).astype(str)
    )

summary = summary.reset_index().rename(columns={"index": "model"})
summary


,model,eval_bleu,eval_chrf,eval_ter,eval_runtime
0,Helsinki-NLP/opus-mt-es-en,45.61 ± 1.05,66.98 ± 1.22,45.78 ± 1.79,129.04 ± 9.5
1,facebook/m2m100_1.2B,40.76 ± 0.92,63.02 ± 1.19,49.48 ± 1.76,870.25 ± 62.98
2,facebook/m2m100_418M,38.71 ± 1.05,61.61 ± 1.25,52.27 ± 1.88,480.35 ± 34.43
3,facebook/nllb-200-distilled-600M,41.12 ± 1.01,64.61 ± 1.28,46.4 ± 1.86,432.22 ± 31.66


In [4]:
finetuned_results_df = pd.read_csv("../output/results_finetuned_all.tsv", sep="\t")
finetuned_results_df.head()

,eval_bleu,eval_chrf,eval_ter,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,model,fold
0,55.838977,71.748367,39.896448,1.053779,138.1946,27.085,1.693,5.0,Helsinki-NLP/opus-mt-es-en,fold_1
1,53.664249,69.422811,42.985668,1.472476,142.5935,28.080,1.760,5.0,Helsinki-NLP/opus-mt-es-en,fold_2
2,56.776534,72.868433,37.555775,0.985740,128.3647,27.025,1.690,5.0,Helsinki-NLP/opus-mt-es-en,fold_3
3,54.836688,71.216674,41.275451,1.095264,147.5500,24.521,1.538,5.0,Helsinki-NLP/opus-mt-es-en,fold_4
4,55.335273,72.032263,40.150704,1.032413,121.8741,27.184,1.707,5.0,Helsinki-NLP/opus-mt-es-en,fold_5


In [5]:
metric_cols = ["eval_bleu","eval_chrf","eval_ter","eval_runtime"]

agg = finetuned_results_df.groupby("model")[metric_cols].agg(["mean","std"])

finetuned_summary = pd.DataFrame(index=agg.index)

for metric in metric_cols:
    finetuned_summary[metric] = (
        agg[(metric, "mean")].round(2).astype(str)
        + " ± "
        + agg[(metric, "std")].round(2).astype(str)
    )

finetuned_summary = finetuned_summary.reset_index().rename(columns={"index": "model"})
finetuned_summary["model"] = finetuned_summary["model"].apply(lambda x: str(x)+"-finetuned")
finetuned_summary

,model,eval_bleu,eval_chrf,eval_ter,eval_runtime
0,Helsinki-NLP/opus-mt-es-en-finetuned,55.29 ± 1.16,71.46 ± 1.28,40.37 ± 1.99,135.72 ± 10.48


In [6]:
complete_summary = pd.concat([finetuned_summary, summary], ignore_index=True)
complete_summary = complete_summary.rename(
    columns={
        "model": "Model",
        "eval_bleu": "BLEU",
        "eval_chrf": "CHRF",
        "eval_ter": "TER"
    }
)[["Model", "BLEU", "CHRF", "TER"]]

complete_summary

,Model,BLEU,CHRF,TER
0,Helsinki-NLP/opus-mt-es-en-finetuned,55.29 ± 1.16,71.46 ± 1.28,40.37 ± 1.99
1,Helsinki-NLP/opus-mt-es-en,45.61 ± 1.05,66.98 ± 1.22,45.78 ± 1.79
2,facebook/m2m100_1.2B,40.76 ± 0.92,63.02 ± 1.19,49.48 ± 1.76
3,facebook/m2m100_418M,38.71 ± 1.05,61.61 ± 1.25,52.27 ± 1.88
4,facebook/nllb-200-distilled-600M,41.12 ± 1.01,64.61 ± 1.28,46.4 ± 1.86


In [7]:
complete_summary.to_csv("../output/complete_summary.tsv", sep="\t", index=False)